# Soft-wall quenched log-polar effective Hamiltonian

This local CPU notebook analyzes all 300 completed soft-v3 trajectories in the full $40N_y$ transfer sector. It constructs $A_\xi=\operatorname{arctanh}G_\xi$ before the $y$ twirl and disorder average, compares quenched and typical generators, and tests wall chirality separately from Cardy--Calabrese criticality.

In [1]:
# Runtime and complete editable analysis configuration. Run this cell first.
import os
from pathlib import Path

CPU_RANGE = ''                 # Example: '0-15' or '0-7,16-23'.
N_WORKERS = 1                 # Parallel trajectories within a five-sample shard.
NY_VALUES = (20, 30, 40)
MAX_SAMPLES_PER_NY = None     # Set to 1 for smoke; None uses all 100 per size.
VERIFY_SOURCE_HASHES = True
POLAR_LOG_CAPS = (8.0, 10.0, 12.0)
DEFAULT_POLAR_LOG_CAP = 10.0
RENYI_ORDERS = (1, 2, 3)
BOUNDARY_TWISTS = (-1.0e-7, 1.0e-7)
BOOTSTRAP_REPLICATES = 2000
JACKKNIFE_GROUPS = 10
OUTPUT_REVISION = 'soft_quenched_log_polar_v1'
OUTPUT_ROOT_OVERRIDE = ''     # Absolute or repo-relative path; blank uses this bundle.

def _parse_cpu_range(spec):
    selected = set()
    for token in spec.replace(' ', '').split(','):
        if not token:
            continue
        if '-' in token:
            lo, hi = (int(value) for value in token.split('-', 1))
            selected.update(range(lo, hi + 1))
        else:
            selected.add(int(token))
    return selected

available_cpus = set(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else set()
if CPU_RANGE and available_cpus:
    requested_cpus = _parse_cpu_range(CPU_RANGE)
    if not requested_cpus or not requested_cpus.issubset(available_cpus):
        raise ValueError(f'CPU_RANGE={CPU_RANGE!r} is not contained in {sorted(available_cpus)}')
    os.sched_setaffinity(0, requested_cpus)
allocated_cpus = sorted(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else []
blas_threads = 1 if N_WORKERS > 1 else max(1, len(allocated_cpus))
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = str(blas_threads)
print({'allocated_cpus': allocated_cpus, 'workers': N_WORKERS, 'blas_threads': blas_threads})

{'allocated_cpus': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], 'workers': 1, 'blas_threads': 12}


## Theory and estimator order

For every disorder trajectory, the left polar convention is $T_\xi=P_{L,\xi}U_\xi$, $A_\xi=\log P_{L,\xi}=\operatorname{arctanh}G_\xi$, and $h_\xi=-2A_\xi$. The nonlinear logarithm is taken trajectory by trajectory, then each result is twirled along $y$, and only then is the quenched mean formed. Soft walls retain all $x=0,\ldots,19$ columns; wall weights use $(4,5,6)$ and $(14,15,16)$ without truncation. Cardy--Calabrese scaling tests central charge, while localized counterpropagating branches test chirality.

In [2]:
import json
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

candidate = Path.cwd()
if not (candidate / 'polar_effective_hamiltonian.py').is_file():
    candidate = Path('00_WORKSPACE/CURRENT/final_production_new_designs/07_maxmix_hard_soft_purification').resolve()
BUNDLE_ROOT = candidate
if not (BUNDLE_ROOT / 'polar_effective_hamiltonian.py').is_file():
    raise FileNotFoundError('Could not locate campaign 07')
sys.path.insert(0, str(BUNDLE_ROOT))
import polar_effective_hamiltonian as polar

SPEC = polar.SOFT_SPEC
DATA_ROOT = SPEC.data_root
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve() if OUTPUT_ROOT_OVERRIDE else BUNDLE_ROOT / 'analysis_outputs' / OUTPUT_REVISION
CACHE_ROOT = OUTPUT_ROOT / 'sample_cache'
FIGURE_ROOT = OUTPUT_ROOT / 'figure_assets'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = polar.AnalysisConfig(construction='soft', caps=POLAR_LOG_CAPS, default_cap=DEFAULT_POLAR_LOG_CAP, renyi_orders=RENYI_ORDERS, twists=BOUNDARY_TWISTS, bootstrap_replicates=BOOTSTRAP_REPLICATES, jackknife_groups=JACKKNIFE_GROUPS)
mpl.rcParams.update({'font.family': 'CMU Sans Serif', 'font.size': 8, 'axes.linewidth': 0.8})
print(json.dumps({'data_root': str(DATA_ROOT), 'output_root': str(OUTPUT_ROOT), 'analysis_config': CONFIG.identity()}, indent=2))

{
  "data_root": "/home/abhuiyan/class_A_fermionic_adaptive_circuit/00_WORKSPACE/CURRENT/final_production_new_designs/07_maxmix_hard_soft_purification/gpu_data/maxmix_hard_soft_purification_nx20_ny20-40_s100_4ny_raster_v3/soft",
  "output_root": "/home/abhuiyan/class_A_fermionic_adaptive_circuit/00_WORKSPACE/CURRENT/final_production_new_designs/07_maxmix_hard_soft_purification/analysis_outputs/soft_quenched_log_polar_v1",
  "analysis_config": {
    "analysis_schema": "domain_wall_quenched_log_polar_analysis_v2",
    "caps": [
      8.0,
      10.0,
      12.0
    ],
    "default_cap": 10.0,
    "renyi_orders": [
      1,
      2,
      3
    ],
    "twists": [
      -1e-07,
      1e-07
    ],
    "fit_ay_min": 8,
    "sensitivity_ay_min": 6,
    "bootstrap_replicates": 2000,
    "bootstrap_seed": 2026091907,
    "jackknife_groups": 10,
    "wall_weight_min": 0.5,
    "construction": "soft",
    "source_revision": "maxmix_hard_soft_purification_nx20_ny20-40_s100_4ny_raster_v3",
    "act

## Exact coupled-domain-wall calibration

Before touching campaign data, reproduce the accepted untruncated exact-B0 estimator calibration at $N_x=20,N_y=48$.

In [3]:
B0_TABLE = BUNDLE_ROOT.parents[1] / 'experiment_review' / 'b0_exact_domain_wall' / 'results' / '20260816_183928' / 'processed' / 'tables' / 'entropy_fits.csv'
b0 = pd.read_csv(B0_TABLE)
b0_check = b0[(b0.construction == 'coupled') & (b0.geometry_key == 'coupled__Nx020__Ny048') & (b0.quantity == 'full_strip')].copy()
if set(b0_check.q) != {1, 2, 3} or np.max(np.abs(b0_check.c_estimate - 1.0)) >= 0.02:
    raise RuntimeError('Exact coupled B0 calibration did not reproduce c_w=1')
display(b0_check[['q', 'slope', 'c_estimate', 'r2', 'ay_min', 'ay_max']])

,q,slope,c_estimate,r2,ay_min,ay_max
45,1,0.333364,1.000092,1.000000,8,24
48,2,0.248978,0.995910,0.999999,8,24
51,3,0.220747,0.993360,0.999998,8,24


## Verify the completed soft-v3 inventory

All result/completion pairs are checked before analysis. The displayed contract is read from the identity that was verified against every shard.

In [4]:
records = polar.discover_samples(DATA_ROOT, construction='soft', ny_values=NY_VALUES, verify_hashes=VERIFY_SOURCE_HASHES, require_complete=True)
inventory = pd.DataFrame([{'Ny': row.ny, 'sample': row.sample_index, 'source': row.result_path.name} for row in records])
display(inventory.groupby('Ny').agg(samples=('sample', 'count'), first=('sample', 'min'), last=('sample', 'max')))
campaign_parameters = {'sampling_revision': SPEC.revision, 'configuration_hash': SPEC.configuration_hash, 'construction': SPEC.construction, 'Nx': polar.EXPECTED_NX, 'Ny': list(NY_VALUES), 'cycles': {str(ny): 4 * ny for ny in NY_VALUES}, 'dtype': 'complex128', 'active_x': list(SPEC.active_x), 'active_dimension': {str(ny): SPEC.transfer_mode_count(ny) for ny in NY_VALUES}, 'trajectory_count': len(records)}
print(json.dumps(campaign_parameters, indent=2))

,samples,first,last
Ny,,,
20,100,0,99
30,100,0,99
40,100,0,99


{
  "sampling_revision": "maxmix_hard_soft_purification_nx20_ny20-40_s100_4ny_raster_v3",
  "configuration_hash": "a0169e31d63dc77f08da34b491a88516a05cdb1dd4f98badd19e1735c180b821",
  "construction": "soft",
  "Nx": 20,
  "Ny": [
    20,
    30,
    40
  ],
  "cycles": {
    "20": 80,
    "30": 120,
    "40": 160
  },
  "dtype": "complex128",
  "active_x": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19
  ],
  "active_dimension": {
    "20": 800,
    "30": 1200,
    "40": 1600
  },
  "trajectory_count": 300
}


## Build or resume trajectory caches

Each checksum-bound cache is committed with a completion JSON. Restarting skips valid trajectories and repeats only an interrupted trajectory.

In [5]:
cache_report = polar.build_sample_caches(records, cache_root=CACHE_ROOT, config=CONFIG, progress=tqdm, max_samples_per_ny=MAX_SAMPLES_PER_NY, workers=N_WORKERS)
print('[cache completion]', json.dumps(cache_report, indent=2))

log-polar trajectories:   0%|          | 0/300 [00:00<?, ?trajectory/s]

[cache completion] {
  "selected": 300,
  "completed": 300,
  "skipped": 0
}


## Aggregate quenched, typical, annealed, and flattened analyses

In [6]:
analysis = polar.aggregate_analysis(records, cache_root=CACHE_ROOT, output_root=OUTPUT_ROOT, config=CONFIG, max_samples_per_ny=MAX_SAMPLES_PER_NY)
summary = analysis['summary']
spectrum_df = pd.read_csv(OUTPUT_ROOT / 'spectrum_rows.csv')
velocity_df = pd.read_csv(OUTPUT_ROOT / 'velocity_summary.csv')
entropy_df = pd.read_csv(OUTPUT_ROOT / 'entropy_curves.csv')
fit_df = pd.read_csv(OUTPUT_ROOT / 'entropy_fit_summary.csv')
diagnostics_df = pd.read_csv(OUTPUT_ROOT / 'numerical_diagnostics.csv')
arrays = np.load(OUTPUT_ROOT / 'notebook_arrays.npz', allow_pickle=False)
print(json.dumps(summary['claim_ledger'], indent=2))

{
  "numerical": {
    "pass": true,
    "maximum_residuals": {
      "full_hermiticity_residual": 0.0,
      "active_hermiticity_residual": 0.0,
      "active_exterior_coupling": 0.0,
      "exterior_purity_residual": 0.0,
      "exterior_inter_y_coupling": 0.0,
      "g_twirl_hermiticity_residual": 1.6667795129466247e-16,
      "g_twirl_translation_residual": 0.0,
      "g_fourier_reconstruction_residual": 3.3306690738754696e-16,
      "polar_hermiticity_residual": 0.0,
      "a_twirl_hermiticity_residual": 2.6645352591003757e-15,
      "a_twirl_translation_residual": 0.0,
      "a_fourier_reconstruction_residual": 5.329070518200751e-15,
      "projector_idempotency_residual": 4.070506550292025e-15,
      "half_filling_charge_residual": 2.2737367544323206e-13,
      "entropy_origin_spread": 9.121592370320286e-12
    }
  },
  "edge_modes": {
    "release": true,
    "by_size": {
      "20": {
        "caps": [
          {
            "cap": 8.0,
            "walls": [
              {


## Quenched and typical wall spectra

The same overlap-tracked branches are shown as the endpoint generator and as the per-cycle rate.

In [7]:
colors = ('#d62728', '#1f77b4')
markers = ('^', 'o')
fig, axes = plt.subplots(2, len(NY_VALUES), figsize=(7.05, 4.35), sharex='col')
for panel, ny in enumerate(NY_VALUES):
    key = f'Ny{ny:03d}_cap{DEFAULT_POLAR_LOG_CAP:g}'
    ky = arrays[f'{key}_quenched_ky']; low = arrays[f'{key}_quenched_low_energy']
    qwall = arrays[f'{key}_quenched_wall_energy']; twall = arrays[f'{key}_typical_wall_energy_mean']
    for row, scale in enumerate((1.0, 1.0 / (4 * ny))):
        ax = axes[row, panel]
        for band in range(low.shape[1]): ax.plot(ky, scale * low[:, band], '.', ms=2, color='0.72')
        for wall in range(2):
            ax.plot(ky, scale * qwall[:, wall], color=colors[wall], marker=markers[wall], ms=2.7, lw=1, label=f'wall {wall+1}, quenched')
            ax.plot(ky, scale * twall[:, wall], color=colors[wall], ls='--', lw=0.9, label=f'wall {wall+1}, typical')
        ax.axhline(0, color='black', ls=':', lw=0.8); ax.tick_params(direction='in', top=True, right=True)
    axes[0, panel].set_title(rf'$N_y={ny}$'); axes[1, panel].set_xlabel(r'$k_y$')
axes[0, 0].set_ylabel(r'$h_{\mathrm{eff}}=-2\log P_L$'); axes[1, 0].set_ylabel(r'$h_{\mathrm{eff}}/(4N_y)$')
axes[0, -1].legend(frameon=False, fontsize=5.5); fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'soft_quenched_typical_wall_spectrum.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_quenched_typical_wall_spectrum.png', dpi=300, bbox_inches='tight')
plt.show()

## Localized branch dispersions and velocities

Opposite velocity signs at branches carrying at least 50% weight in the full three-column wall windows are the chirality criterion.

In [8]:
fig, axes = plt.subplots(1, len(NY_VALUES), figsize=(7.05, 2.35), sharey=True)
for panel, (ax, ny) in enumerate(zip(np.atleast_1d(axes), NY_VALUES)):
    for ci, cap in enumerate(POLAR_LOG_CAPS):
        velocity = arrays[f'Ny{ny:03d}_cap{cap:g}_quenched_velocity']
        for wall in range(2): ax.plot(cap + (wall - .5) * .10, velocity[wall], marker=markers[wall], color=colors[wall], ms=5, ls='none', label=f'wall {wall+1}' if ci == 0 else None)
    ax.axhline(0, color='black', ls='--', lw=.8); ax.set_title(rf'$N_y={ny}$'); ax.set_xlabel(r'polar cap $\Lambda$'); ax.tick_params(direction='in', top=True, right=True)
axes[0].set_ylabel(r'$\partial h_{\mathrm{eff}}/\partial k_y$'); axes[-1].legend(frameon=False, fontsize=7); fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'soft_wall_velocity_cap_sensitivity.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_wall_velocity_cap_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

## Cardy--Calabrese entropy curves

Whole-trajectory intervals accompany endpoint and typical curves; quenched-mean uncertainty is supplied by ten-group delete-group jackknife.

In [9]:
family_style = {'actual_endpoint': ('#1f77b4', 'o', '-', 'actual endpoint'), 'typical_sample': ('#2ca02c', 's', '--', 'typical ground state'), 'quenched_mean': ('#d62728', '^', ':', 'quenched ground state')}
fig, axes = plt.subplots(1, len(NY_VALUES), figsize=(7.05, 2.55))
for ax, ny in zip(np.atleast_1d(axes), NY_VALUES):
    for family, (color, marker, linestyle, label) in family_style.items():
        sub = entropy_df[(entropy_df.family == family) & (entropy_df.Ny == ny) & (entropy_df.renyi_order == 1)]
        if family != 'actual_endpoint': sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
        sub = sub.sort_values('Ay'); x = polar.log_chord(ny, sub.Ay.to_numpy())
        ax.plot(x, sub['mean'], color=color, marker=marker, ms=3, lw=1, ls=linestyle, label=label)
        ax.fill_between(x, sub.ci_low, sub.ci_high, color=color, alpha=.10, lw=0)
    ax.set_title(rf'$N_y={ny}$'); ax.set_xlabel(r'$\log[(N_y/\pi)\sin(\pi A_y/N_y)]$'); ax.tick_params(direction='in', top=True, right=True)
axes[0].set_ylabel(r'$S_1(A_y)$'); axes[-1].legend(frameon=False, fontsize=6); fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'soft_cardy_calabrese_entropy_curves.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_cardy_calabrese_entropy_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## Extracted central charge by Renyi order

In [10]:
fig, axes = plt.subplots(1, len(RENYI_ORDERS), figsize=(7.05, 2.45), sharey=True)
for ax, order in zip(np.atleast_1d(axes), RENYI_ORDERS):
    for family, (color, marker, linestyle, label) in family_style.items():
        sub = fit_df[(fit_df.family == family) & (fit_df.renyi_order == order) & (fit_df.fit_variant == 'primary')]
        if family != 'actual_endpoint': sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
        sub = sub.sort_values('Ny'); central = sub.c_per_wall.to_numpy(dtype=float)
        low = pd.to_numeric(sub.ensemble_c_ci_low).to_numpy(dtype=float); high = pd.to_numeric(sub.ensemble_c_ci_high).to_numpy(dtype=float)
        yerr = np.vstack((np.maximum(0, central - low), np.maximum(0, high - central)))
        ax.errorbar(sub.Ny, central, yerr=yerr, color=color, marker=marker, ms=4, lw=1, ls=linestyle, capsize=2, label=label)
    ax.axhline(1, color='black', ls='--', lw=.8); ax.set_title(rf'$n={order}$'); ax.set_xlabel(r'$N_y$'); ax.set_xticks(NY_VALUES); ax.tick_params(direction='in', top=True, right=True)
axes[0].set_ylabel(r'$c_{w,n}$'); axes[-1].legend(frameon=False, fontsize=6); fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'soft_renyi_central_charge_comparison.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_renyi_central_charge_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Cap, estimator-order, fit-window, and twist controls

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(7.05, 5.0)); ny = max(NY_VALUES)
ax = axes[0, 0]
for cap in POLAR_LOG_CAPS:
    sub = fit_df[(fit_df.family == 'quenched_mean') & (fit_df.Ny == ny) & (fit_df.renyi_order == 1) & (fit_df.fit_variant == 'primary')]
    sub = sub[np.isclose(pd.to_numeric(sub.cap), cap) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
    if len(sub): ax.plot(cap, sub.iloc[0].c_per_wall, 'o', color='#d62728')
ax.set_title('polar-cap sensitivity'); ax.set_xlabel(r'$\Lambda$'); ax.set_ylabel(r'$c_{w,1}$')
ax = axes[0, 1]
for family, label in (('quenched_mean', 'quenched'), ('annealed_covariance', 'annealed'), ('flattened_parent', 'flattened')):
    sub = fit_df[(fit_df.family == family) & (fit_df.Ny == ny) & (fit_df.renyi_order == 1) & (fit_df.fit_variant == 'primary')]
    if family == 'quenched_mean': sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
    if len(sub): ax.plot([label], [sub.iloc[0].c_per_wall], 'o')
ax.set_title('estimator-order controls'); ax.tick_params(axis='x', rotation=15)
ax = axes[1, 0]; variants = ('primary', 'aymin6', 'drop_endpoint')
for family, (color, marker, _, label) in family_style.items():
    sub = fit_df[(fit_df.family == family) & (fit_df.Ny == ny) & (fit_df.renyi_order == 1)]
    if family != 'actual_endpoint': sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP) & np.isclose(pd.to_numeric(sub.twist), max(BOUNDARY_TWISTS))]
    ax.plot(variants, [sub[sub.fit_variant == variant].iloc[0].c_per_wall for variant in variants], marker=marker, color=color, label=label)
ax.set_title('fit-window sensitivity'); ax.tick_params(axis='x', rotation=15)
ax = axes[1, 1]
for family, color, marker in (('quenched_mean', '#d62728', '^'), ('typical_sample', '#2ca02c', 's')):
    sub = fit_df[(fit_df.family == family) & (fit_df.Ny == ny) & (fit_df.renyi_order == 1) & (fit_df.fit_variant == 'primary')]
    sub = sub[np.isclose(pd.to_numeric(sub.cap), DEFAULT_POLAR_LOG_CAP)].sort_values('twist')
    ax.plot(sub.twist * 1e7, sub.c_per_wall, marker=marker, color=color, label=family)
ax.set_title('twist sensitivity'); ax.set_xlabel(r'$\phi\times10^7$'); ax.legend(frameon=False, fontsize=6)
for ax in axes.flat: ax.axhline(1, color='black', ls='--', lw=.8); ax.tick_params(direction='in', top=True, right=True)
fig.tight_layout(); fig.savefig(FIGURE_ROOT / 'soft_cap_estimator_fit_twist_controls.pdf', bbox_inches='tight')
fig.savefig(FIGURE_ROOT / 'soft_cap_estimator_fit_twist_controls.png', dpi=300, bbox_inches='tight'); plt.show()

## Raw diagnostics and claim ledger

This final cell exposes fit rows, numerical maxima, saturation counts, and all gated interpretations.

In [12]:
display(fit_df.sort_values(['family', 'Ny', 'renyi_order', 'fit_variant']).reset_index(drop=True))
numeric_columns = [column for column in diagnostics_df.columns if column.endswith('residual') or column.endswith('coupling')]
display(diagnostics_df[numeric_columns].max(numeric_only=True).to_frame('maximum'))
display(diagnostics_df.groupby(['family', 'Ny', 'cap'], dropna=False).agg(rows=('Ny', 'size'), interior_modes=('interior_mode_count', 'median'), capped_modes=('capped_mode_count', 'median')).reset_index())
display(velocity_df)
print(json.dumps(summary['claim_ledger'], indent=2))
print(f'analysis complete: {OUTPUT_ROOT}')

,family,Ny,samples,cap,twist,renyi_order,fit_variant,slope,intercept,r_squared,c_per_wall,n_points,ensemble_c_mean,ensemble_c_ci_low,ensemble_c_ci_high
0,actual_endpoint,20,100,NaN,NaN,1,aymin6,0.386917,9.001364,0.999975,1.160751,5,1.160751,1.137441,1.186291
1,actual_endpoint,20,100,NaN,NaN,1,drop_endpoint,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
2,actual_endpoint,20,100,NaN,NaN,1,primary,0.382754,9.008950,0.999998,1.148263,3,1.148263,1.122712,1.177366
3,actual_endpoint,20,100,NaN,NaN,2,aymin6,0.287517,6.796921,0.999979,1.150068,5,1.150068,1.129342,1.172636
4,actual_endpoint,20,100,NaN,NaN,2,drop_endpoint,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400,typical_sample,40,100,8.0,1.000000e-07,3,primary,0.292250,5.760093,0.998160,1.315124,13,1.315124,1.128590,1.509105
401,typical_sample,40,100,10.0,-1.000000e-07,3,primary,0.292163,5.760206,0.998159,1.314732,13,1.314732,1.129384,1.530602
402,typical_sample,40,100,10.0,1.000000e-07,3,primary,0.292163,5.760206,0.998159,1.314732,13,1.314732,1.127228,1.533384
403,typical_sample,40,100,12.0,-1.000000e-07,3,primary,0.283401,5.776422,0.998525,1.275306,13,1.275306,1.118078,1.493586


,maximum
projector_idempotency_residual,4.070507e-15
half_filling_charge_residual,2.273737e-13
active_exterior_coupling,0.000000e+00
active_hermiticity_residual,0.000000e+00
exterior_inter_y_coupling,0.000000e+00
exterior_purity_residual,0.000000e+00
full_hermiticity_residual,0.000000e+00
g_fourier_reconstruction_residual,3.330669e-16
g_twirl_hermiticity_residual,1.666780e-16
g_twirl_translation_residual,0.000000e+00


,family,Ny,cap,rows,interior_modes,capped_modes
0,annealed_covariance,20,10.0,1,NaN,NaN
1,annealed_covariance,30,10.0,1,NaN,NaN
2,annealed_covariance,40,10.0,1,NaN,NaN
3,flattened_parent,20,10.0,1,NaN,NaN
4,flattened_parent,30,10.0,1,NaN,NaN
5,flattened_parent,40,10.0,1,NaN,NaN
6,quenched_mean,20,8.0,2,NaN,NaN
7,quenched_mean,20,10.0,2,NaN,NaN
8,quenched_mean,20,12.0,2,NaN,NaN
9,quenched_mean,30,8.0,2,NaN,NaN


,method,Ny,samples,cap,wall,crossing,velocity,velocity_rate,velocity_r_squared,crossing_wall_weight,velocity_ci_low,velocity_ci_high
0,quenched_mean,20,100,8.0,0,-0.053704,7.975678,0.099696,0.999300,0.999769,NaN,NaN
1,quenched_mean,20,100,8.0,1,-0.058366,-9.215519,-0.115194,0.995221,0.999577,NaN,NaN
2,typical_sample,20,100,8.0,0,-0.023011,9.057460,0.113218,NaN,0.999597,7.845068,10.179941
3,typical_sample,20,100,8.0,1,-0.074135,-11.556733,-0.144459,NaN,0.898709,-12.321875,-10.826002
4,quenched_mean,20,100,10.0,0,-0.051696,9.978606,0.124733,0.999377,0.999769,NaN,NaN
5,quenched_mean,20,100,10.0,1,-0.057824,-11.527256,-0.144091,0.995205,0.999574,NaN,NaN
6,typical_sample,20,100,10.0,0,-0.024464,11.427219,0.142840,NaN,0.999605,10.011404,12.747821
7,typical_sample,20,100,10.0,1,-0.074135,-14.444881,-0.180561,NaN,0.898724,-15.328471,-13.542796
8,quenched_mean,20,100,12.0,0,-0.049098,11.985632,0.149820,0.999478,0.999769,NaN,NaN
9,quenched_mean,20,100,12.0,1,-0.056648,-13.839113,-0.172989,0.995349,0.999572,NaN,NaN


{
  "numerical": {
    "pass": true,
    "maximum_residuals": {
      "full_hermiticity_residual": 0.0,
      "active_hermiticity_residual": 0.0,
      "active_exterior_coupling": 0.0,
      "exterior_purity_residual": 0.0,
      "exterior_inter_y_coupling": 0.0,
      "g_twirl_hermiticity_residual": 1.6667795129466247e-16,
      "g_twirl_translation_residual": 0.0,
      "g_fourier_reconstruction_residual": 3.3306690738754696e-16,
      "polar_hermiticity_residual": 0.0,
      "a_twirl_hermiticity_residual": 2.6645352591003757e-15,
      "a_twirl_translation_residual": 0.0,
      "a_fourier_reconstruction_residual": 5.329070518200751e-15,
      "projector_idempotency_residual": 4.070506550292025e-15,
      "half_filling_charge_residual": 2.2737367544323206e-13,
      "entropy_origin_spread": 9.121592370320286e-12
    }
  },
  "edge_modes": {
    "release": true,
    "by_size": {
      "20": {
        "caps": [
          {
            "cap": 8.0,
            "walls": [
              {
